In [1]:
import tensorflow as tf 
import numpy as np

ds = tf.data.Dataset.range(10) # делаем датасет из чисел от 0 до 9
ds = ds.map(lambda x: x * 2) # применяем функцию к каждому элементу
ds = ds.batch(3) # собираем элементы в батчи по три штуки

for batch in ds.take(2): # берем первые 2 батча
    print(batch.numpy())

I0000 00:00:1793636409.559915    5537 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE3 SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


[0 2 4]
[ 6  8 10]


In [7]:
ds = tf.data.Dataset.range(10).shuffle(buffer_size=10).batch(5)

for epoch in range(2):
    print(f"Epoch: {epoch + 1}")
    for batch in ds:
        print(batch.numpy())

Epoch: 1
[6 5 8 0 3]
[1 2 9 7 4]
Epoch: 2
[7 9 3 6 5]
[1 4 8 0 2]


In [9]:
images = tf.random.uniform((12, 28, 28, 1), maxval=255, minval=1, dtype=tf.float32)
labels = tf.range(12) % 10

def preprocess(image, label):
    image = tf.image.resize(image, (32, 32)) # меняем размер изображения
    image = image / 255.0 # нормализуем пиксели 
    image = tf.image.random_flip_left_right(image) # случайное отражение
    return image, label

ds = (
    tf.data.Dataset.from_tensor_slices((images, labels))
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE) # позволяю тф самому выбрать колво параллельных операций
    .batch(4)
    .prefetch(tf.data.AUTOTUNE)
)

for x, y in ds.take(1):
    print(x.shape, y.numpy())

(4, 32, 32, 1) [0 1 2 3]


In [ ]:
dataset = (
    tf.data.Dataset.from_tensor_slices((X, y))
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .cache()
    .shuffle(buffer_size=1000)
    .batch(32)
    .prefetch(tf.data.AUTOTUNE)
)

In [11]:
def make_dataset(X, y, batch_size=32, training=False):
    ds = tf.data.Dataset.from_tensor_slices((X, y))

    def preprocess(x, label):
        x = tf.cast(x, tf.float32) / 255.0
        return x, label

    ds = ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)

    if training:
        ds = ds.cache()
        shuffle_buffer = min(10_000, len(X))
        ds = ds.shuffle(buffer_size=shuffle_buffer, reshuffle_each_iteration=True)
    else:
        ds = ds.cache()

    ds = ds.batch(batch_size)
    ds = ds.prefetch(tf.data.AUTOTUNE)

    return ds

In [12]:
X = np.random.randint(0, 256, size=(200, 28, 28, 1)).astype(np.float32)
y = np.random.randint(0, 10, size=(200,)).astype(np.int64)

train_ds = make_dataset(X[:160], y[:160], batch_size=16, training=True)
val_ds = make_dataset(X[160:], y[160:], batch_size=16, training=False)

In [13]:
model = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=(28, 28, 1)),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(10),
])

model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy'],
)

/usr/lib/python3.14/site-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [14]:
history = model.fit(
    train_ds,
    epochs=2,
    validation_data=val_ds,
    verbose=2
)

Epoch 1/2


/usr/lib/python3.14/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


10/10 - 2s - 173ms/step - accuracy: 0.0938 - loss: 2.5094 - val_accuracy: 0.1000 - val_loss: 2.3946
Epoch 2/2
10/10 - 0s - 7ms/step - accuracy: 0.1000 - loss: 2.3248 - val_accuracy: 0.0500 - val_loss: 2.3656


In [17]:
ds = tf.data.Dataset.range(20)
ds = ds.map(lambda x: x  *2)
ds = ds.batch(5)

for batch in ds.take(2):
    print(batch.numpy())

[0 2 4 6 8]
[10 12 14 16 18]


In [4]:
dataset_1 = tf.data.Dataset.range(20).shuffle(buffer_size=5).batch(4)
dataset_2 = tf.data.Dataset.range(20).shuffle(buffer_size=20).batch(4)

for batch_1 in dataset_1:
    print(batch_1.numpy())

print('========================================')
for batch_2 in dataset_2:
    print(batch_2.numpy())

[2 3 4 1]
[6 8 7 9]
[12  0 14 10]
[15 13 11  5]
[17 19 18 16]
[ 9 12 17  1]
[11 15 18  6]
[7 3 2 4]
[13  5 14  0]
[16 10 19  8]


In [8]:
images = tf.random.uniform((16, 28, 28, 1), maxval=255, minval=1, dtype=tf.float32)
labels = tf.range(16) % 10

def image_preprocess(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    image = tf.image.random_brightness(image, max_delta=0.2)
    image = tf.image.random_flip_left_right(image)
    return image, label

ds = (
    tf.data.Dataset.from_tensor_slices((images, labels))
    .map(image_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(4)
    .prefetch(tf.data.AUTOTUNE)
)

for x, y in ds.take(1):
    print(x.shape)
    print(y.numpy())

(4, 28, 28, 1)
[0 1 2 3]


In [10]:
model = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape = (28, 28, 1)),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(10),
])

model.compile(
    optimizer = 'adam',
    loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy'],
)

model.fit(ds, epochs=2, verbose = 2)

Epoch 1/2


/usr/lib/python3.14/site-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/usr/lib/python3.14/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


4/4 - 1s - 326ms/step - accuracy: 0.0625 - loss: 2.4868
Epoch 2/2
4/4 - 0s - 16ms/step - accuracy: 0.1875 - loss: 2.2658


In [3]:
# сжатие файлов

# запись
options = tf.io.TFRecordOptions(compression_type='GZIP')

with tf.io.TFRecordWriter('m_d.tfrecord.gz', options=options) as f:
    f.write(b"first rec")
    f.write(b"second rec")

In [5]:
dataset = tf.data.TFRecordDataset(
    ['m_d.tfrecord.gz'],
    compression_type='GZIP'
)

for record in dataset:
    print(record.numpy())

b'first rec'
b'second rec'


In [6]:
from tensorflow.train import Example, Features, Feature, BytesList, Int64List, FloatList

person_example = Example(
    features=Features(
        feature={
            "name": Feature(
                bytes_list=BytesList(value=[b"Aurelien Geron"])
            ),
            "id": Feature(
                int64_list=Int64List(value=[2345222])
            ),
            "scores": Feature(
                float_list=FloatList(value=[95.5, 88.0, 92.3])
            ),
            "emails": Feature(
                bytes_list=BytesList(value=[b"a@example.com", b"b@example.com"])
            ),
        }
    )
)

print(person_example)

features {
  feature {
    key: "scores"
    value {
      float_list {
        value: 95.5
        value: 88
        value: 92.3
      }
    }
  }
  feature {
    key: "name"
    value {
      bytes_list {
        value: "Aurelien Geron"
      }
    }
  }
  feature {
    key: "id"
    value {
      int64_list {
        value: 2345222
      }
    }
  }
  feature {
    key: "emails"
    value {
      bytes_list {
        value: "a@example.com"
        value: "b@example.com"
      }
    }
  }
}



In [7]:
with tf.io.TFRecordWriter("people.tfrecord") as f:
    f.write(person_example.SerializeToString())

In [ ]:
# чтение обратно
feature_description = {
    "name": tf.io.FixedLenFeature([], tf.string, default_value=""),
    "id": tf.io.FixedLenFeature([], tf.int64, default_value=0),
    "scores": tf.io.VarLenFeature(tf.float32),
    "emails": tf.io.VarLenFeature(tf.string),
}

for raw_record in tf.data.TFRecordDataset(["people.tfrecord"]):
    example = tf.io.parse_single_example(raw_record, feature_description)
    print(example["name"].numpy())
    print(example["id"].numpy())
    print(example["scores"].values.numpy())

b'Aurelien Geron'
2345222
[95.5 88.  92.3]


In [9]:
from tensorflow.train import (
    SequenceExample, Features, FeatureLists, FeatureList,
    Feature, Int64List
)

context = Features(
    feature={
        "movie_id": Feature(int64_list=Int64List(value=[12345])),
        "title": Feature(bytes_list=BytesList(value=[b"Inception"])),
    }
)

ratings = FeatureList(
    feature=[
        Feature(int64_list=Int64List(value=[5])),
        Feature(int64_list=Int64List(value=[3])),
        Feature(int64_list=Int64List(value=[4])),
        Feature(int64_list=Int64List(value=[5])),
    ]
)

seq_example = SequenceExample(
    context=context,
    feature_lists=FeatureLists(feature_list={"ratings": ratings})
)


with tf.io.TFRecordWriter("movies.tfrecord") as f:
    f.write(seq_example.SerializeToString())

In [10]:
context_features = {
    "movie_id": tf.io.FixedLenFeature([], tf.int64),
    "title": tf.io.FixedLenFeature([], tf.string),
}

sequence_features = {
    "ratings": tf.io.FixedLenSequenceFeature([], tf.int64),
}

for raw_record in tf.data.TFRecordDataset(["movies.tfrecord"]):
    context_parsed, sequence_parsed = tf.io.parse_single_sequence_example(
        raw_record,
        context_features=context_features,
        sequence_features=sequence_features,
    )
    print("Movie:", context_parsed["title"].numpy())
    print("Ratings:", sequence_parsed["ratings"].numpy())

Movie: b'Inception'
Ratings: [5 3 4 5]
